In [1]:
from soundevent import data, io


In [3]:
annotation_project = io.load("/Users/ash/Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/whombat-main/save_test/annotation-project-817f6969-580d-4701-9984-c97892207450.json")


In [12]:
# See how many annotation tasks are in the project
print("Annotation tasks ", len(annotation_project.tasks))

# Check the number of annotation tags used
print("Annotation tags: ", len(annotation_project.annotation_tags))

Annotation tasks  31
Annotation tags:  120


In [13]:
def task_is_complete(task: data.AnnotationTask) -> bool:
    """Check if an annotation task is complete.

    A task is considered complete if it has a 'completed' status badge
    and does not have a 'rejected' badge (indicating it needs review).
    """
    for badge in task.status_badges:
        if badge.state == data.AnnotationState.rejected:
            # Task needs review, so it's not considered complete.
            return False

        if badge.state == data.AnnotationState.completed:
            # Task is explicitly marked as complete.
            return True

    # If no 'completed' badge is found, the task is not complete.
    return False


# Create a dictionary mapping clip UUIDs to their completion status.
clip_status = {
    task.clip.uuid: task_is_complete(task)
    for task in annotation_project.tasks
}


def clip_annotation_is_complete(annotation: data.ClipAnnotation) -> bool:
    """Check if a clip annotation is complete based on its task status."""
    if annotation.clip.uuid not in clip_status:
        # If the clip is not part of the project's tasks, ignore it.
        return False

    # Return the pre-computed completion status from the clip_status dictionary.
    return clip_status[annotation.clip.uuid]


def clip_annotation_has_issues(annotation: data.ClipAnnotation) -> bool:
    """Check if a clip annotation has any associated issues."""
    return any(note.is_issue for note in annotation.notes)


# Filter the clip annotations to include only those that are complete and have
# no issues.
annotated_clips = [
    annotation
    for annotation in annotation_project.clip_annotations
    if not clip_annotation_has_issues(annotation)
    and clip_annotation_is_complete(annotation)
]

In [19]:
len(annotated_clips)

2

In [35]:
example_annotation = annotated_clips[1]
recording = example_annotation.clip
tag = example_annotation.tags

In [36]:
print(example_annotation)
print(recording)
print(tag)

uuid=UUID('293982af-bb7c-4e6a-973c-ea2443de1808') clip=Clip(uuid=UUID('f92db7ec-40ad-426f-b1c4-d45977c827a3'), recording=Recording(path=PosixPath('Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/test/split_clips/20250518_045000_0_5.wav')), start_time=0.0, end_time=5.0, features=[]) sound_events=[] sequences=[] tags=[Tag(term=Term(label='Common Name'), value='Eurasian Blue Tit'), Tag(term=Term(label='Scientific Taxon Name'), value='Astur gentilis')] notes=[] created_on=datetime.datetime(2026, 2, 24, 12, 58, 0, 730730)
uuid=UUID('f92db7ec-40ad-426f-b1c4-d45977c827a3') recording=Recording(path=PosixPath('Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/test/split_clips/20250518_045000_0_5.wav')) start_time=0.0 end_time=5.0 features=[]
[Tag(term=Term(label='Common Name'), value='Eurasian Blue Tit'), Tag(term=Term(label='Scientific Taxon Name'), value='Astur gentilis')]


In [10]:
for clip_annotation in annotation_project.clip_annotations:
    if not clip_annotation.tags:
        continue
    print(f"Clip annotation ID: {clip_annotation.tags}")
    clip = clip_annotation.clip
    recording = clip.recording
    print(
        f"* Recording {recording.path} [from "
        f"{clip.start_time:.3f}s to {clip.end_time:.3f}s]"
    )
    print(
        f"\t{len(clip_annotation.sound_events)} sound event annotations found"
    )
    for annotation in clip_annotation.sound_events:
        sound_event = annotation.sound_event
        start_time, end_time = sound_event.geometry.coordinates
        print(f"\t+ Sound event from {start_time:.3f}s to {end_time:.3f}s")
        for tag in annotation.tags:
            print(f"\t\t- {tag}")
    print("")

Clip annotation ID: [Tag(term=Term(label='Scientific Taxon Name'), value='Aegithalos caudatus')]
* Recording Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/test/split_clips/20250316_170901_5_10.wav [from 0.000s to 5.000s]
	0 sound event annotations found

Clip annotation ID: [Tag(term=Term(label='Common Name'), value='Eurasian Blue Tit'), Tag(term=Term(label='Scientific Taxon Name'), value='Astur gentilis')]
* Recording Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/test/split_clips/20250518_045000_0_5.wav [from 0.000s to 5.000s]
	0 sound event annotations found



In [7]:
def task_is_complete(task: data.AnnotationTask) -> bool:
    """Check if an annotation task is complete.

    A task is considered complete if it has a 'completed' status badge
    and does not have a 'rejected' badge (indicating it needs review).
    """
    for badge in task.status_badges:
        if badge.state == data.AnnotationState.rejected:
            # Task needs review, so it's not considered complete.
            return False

        if badge.state == data.AnnotationState.completed:
            # Task is explicitly marked as complete.
            return True

    # If no 'completed' badge is found, the task is not complete.
    return False


# Create a dictionary mapping clip UUIDs to their completion status.
clip_status = {
    task.clip.uuid: task_is_complete(task)
    for task in annotation_project.tasks
}


def clip_annotation_is_complete(annotation: data.ClipAnnotation) -> bool:
    """Check if a clip annotation is complete based on its task status."""
    if annotation.clip.uuid not in clip_status:
        # If the clip is not part of the project's tasks, ignore it.
        return False

    # Return the pre-computed completion status from the clip_status dictionary.
    return clip_status[annotation.clip.uuid]


def clip_annotation_has_issues(annotation: data.ClipAnnotation) -> bool:
    """Check if a clip annotation has any associated issues."""
    return any(note.is_issue for note in annotation.notes)


# Filter the clip annotations to include only those that are complete and have
# no issues.
annotated_clips = [
    annotation
    for annotation in annotation_project.clip_annotations
    if not clip_annotation_has_issues(annotation)
    and clip_annotation_is_complete(annotation)
]

In [8]:
print(annotated_clips)

[ClipAnnotation(uuid=UUID('818bbb0e-ac2b-4f6b-b461-d4ba84041ff6'), clip=Clip(uuid=UUID('6e976ff0-b28c-40e8-81ca-c72788fbfb1b'), recording=Recording(path=PosixPath('Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/test/split_clips/20250316_170901_5_10.wav')), start_time=0.0, end_time=5.0, features=[]), sound_events=[], sequences=[], tags=[Tag(term=Term(label='Scientific Taxon Name'), value='Aegithalos caudatus')], notes=[], created_on=datetime.datetime(2026, 2, 24, 12, 58, 0, 730797)), ClipAnnotation(uuid=UUID('293982af-bb7c-4e6a-973c-ea2443de1808'), clip=Clip(uuid=UUID('f92db7ec-40ad-426f-b1c4-d45977c827a3'), recording=Recording(path=PosixPath('Library/CloudStorage/OneDrive-Nexus365/PHD/Wytham_PAM/test/split_clips/20250518_045000_0_5.wav')), start_time=0.0, end_time=5.0, features=[]), sound_events=[], sequences=[], tags=[Tag(term=Term(label='Common Name'), value='Eurasian Blue Tit'), Tag(term=Term(label='Scientific Taxon Name'), value='Astur gentilis')], notes=[], created_on=datet

In [9]:
from uuid import UUID

# Print each clip name and associated tags (robust to different ClipAnnotation shapes)


# Build a mapping from tag UUID/id to tag name if available on the project
tag_map = {}
for t in getattr(annotation_project, "annotation_tags", []) or []:
    # try common attribute names
    if hasattr(t, "uuid"):
        tag_map[getattr(t, "uuid")] = getattr(t, "name", str(getattr(t, "uuid")))
    elif hasattr(t, "id"):
        tag_map[getattr(t, "id")] = getattr(t, "name", str(getattr(t, "id")))
    else:
        # fallback: try to use the object itself as name
        try:
            tag_map[t] = getattr(t, "name", str(t))
        except Exception:
            continue


def _name_from_clip(clip):
    # common clip name fields
    for attr in ("name", "file_name", "filename", "title"):
        if hasattr(clip, attr):
            val = getattr(clip, attr)
            if val:
                return val
    # metadata dict
    meta = getattr(clip, "metadata", None)
    if isinstance(meta, dict):
        for key in ("name", "title", "file_name", "filename"):
            if key in meta and meta[key]:
                return meta[key]
    # fallback to uuid
    return str(getattr(clip, "uuid", clip))


def _extract_tag_name(item):
    # If item is a simple string
    if isinstance(item, str):
        return item
    # If item is a UUID
    if isinstance(item, UUID):
        return tag_map.get(item, str(item))
    # If item is a dict
    if isinstance(item, dict):
        for key in ("name", "tag", "label", "annotation_tag"):
            if key in item and item[key]:
                return str(item[key])
        # maybe contains a uuid/id
        for key in ("uuid", "id"):
            if key in item and item[key] in tag_map:
                return tag_map[item[key]]
        return None
    # If item is an object, try common attributes
    for attr in ("name", "tag", "label", "annotation_tag", "key", "id", "uuid"):
        if hasattr(item, attr):
            val = getattr(item, attr)
            if val:
                # if it's a UUID/id that maps via tag_map
                if isinstance(val, UUID) or val in tag_map:
                    return tag_map.get(val, str(val))
                return str(val)
    # last resort: string representation
    try:
        return str(item)
    except Exception:
        return None


def extract_tags_from_annotation(annotation):
    candidates = [
        "tags",
        "annotation_tags",
        "labels",
        "annotations",
        "instances",
        "objects",
        "boxes",
        "labels",
        "features",
    ]
    found = set()

    # Check attributes on the annotation
    for attr in candidates:
        if not hasattr(annotation, attr):
            continue
        val = getattr(annotation, attr)
        if val is None:
            continue
        # iterable of items
        if isinstance(val, (list, tuple, set)):
            for item in val:
                name = _extract_tag_name(item)
                if name:
                    found.add(name)
        elif isinstance(val, dict):
            # dict values may be tag objects or lists
            for item in val.values():
                if isinstance(item, (list, tuple, set)):
                    for sub in item:
                        name = _extract_tag_name(sub)
                        if name:
                            found.add(name)
                else:
                    name = _extract_tag_name(item)
                    if name:
                        found.add(name)
        else:
            name = _extract_tag_name(val)
            if name:
                found.add(name)

    # Also check nested annotation items (e.g., an annotation.annotations list of objects with .tag)
    # Some ClipAnnotation implementations store labeled items in .annotations or .instances as objects
    for attr in ("annotations", "instances", "objects", "boxes"):
        if not hasattr(annotation, attr):
            continue
        val = getattr(annotation, attr)
        if not val:
            continue
        if isinstance(val, (list, tuple, set)):
            for item in val:
                # item might have a .tag, .label or .annotation_tag attribute
                name = _extract_tag_name(item)
                if name:
                    found.add(name)

    # If nothing found, try to see if the annotation has a direct .tag or .label attribute
    for attr in ("tag", "label", "annotation_tag", "key"):
        if hasattr(annotation, attr):
            name = _extract_tag_name(getattr(annotation, attr))
            if name:
                found.add(name)

    return sorted(found)


# Print results
for ann in annotated_clips:
    clip_name = _name_from_clip(ann.clip)
    tags = extract_tags_from_annotation(ann)
    print(f"{clip_name}: {', '.join(tags) if tags else '(no tags)'}")

6e976ff0-b28c-40e8-81ca-c72788fbfb1b: dwc:scientificName
f92db7ec-40ad-426f-b1c4-d45977c827a3: dwc:scientificName, dwc:vernacularName
